# 🎬 CorridorKey 2.0 — Photo Background Remover

Upload one photo and get a **transparent RGBA PNG** of the foreground. Supported inputs: PNG, JPG, WebP, TIFF, BMP (anything Pillow can read).

**Quick start:** click **Runtime → Run all**, then choose your photo when the file picker appears. The defaults use **INT8 weights + low-VRAM chunking** and cap the longest side at **2048 px** so a free T4 can finish.

**What you need:** a Colab GPU runtime (**Runtime → Change runtime type → Hardware accelerator: GPU**). A free T4 works with the INT8 path and the automatic float16 compatibility patch. The first run installs PyTorch, downloads the **~1.0 GB INT8 checkpoint**, and downloads the foundation models (about 10–15 minutes). Later runs in the same session skip that.

**What happens:** the notebook clones the pinned [CorridorKey 2.0](https://github.com/nikopueringer/CorridorKey2.0) repo, adds the CUDA INT8 ConvRot path, runs `infer.py` on a one-frame directory, and extracts the model's `cutout` output as a PNG with alpha. The subject is chosen automatically unless you provide a hint in the optional section.

<details>
<summary><b>License (short version)</b></summary>

CorridorKey 2.0 was designed and trained by Zachary Harris in collaboration with Niko and Corridor Digital. You may use it and the outputs for personal or commercial work. You may not sell the tool, its code, or its model, and you may not offer it as a paid API/inference service without permission. If you build on it, your release must stay free, keep the same license, and keep the CorridorKey name. Organizations with more than US $20M annual revenue need a plugin seat per user. Full terms: [LICENSE](https://github.com/nikopueringer/CorridorKey2.0/blob/master/LICENSE). Donationware and pay-what-you-want is fine.

</details>


In [ ]:
#@title 1. Check GPU { display-mode: "form" }
import json
import shutil
import subprocess
import sys

smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
    capture_output=True, text=True,
)
print(smi.stdout.strip() or "nvidia-smi did not report a GPU.")

try:
    meminfo = {}
    for line in open("/proc/meminfo"):
        key, value = line.split(":", 1)
        meminfo[key] = value.strip()
    total = int(meminfo["MemTotal"].split()[0]) / 2**20
    available = int(meminfo["MemAvailable"].split()[0]) / 2**20
    print(f"System RAM: {total:.1f} GB total, {available:.1f} GB available")
except Exception as exc:
    print("Could not read system RAM:", exc)

try:
    du = shutil.disk_usage("/content")
    print(f"Colab disk /content: {du.free/2**30:.1f} GB free of {du.total/2**30:.1f} GB")
except Exception as exc:
    print("Could not read disk space:", exc)

code = r"""
import json, torch
p = torch.cuda.get_device_properties(0)
print(json.dumps({"torch": torch.__version__, "name": p.name,
                  "vram_gb": p.total_memory / 1e9, "cc": [p.major, p.minor]}))
"""
try:
    out = subprocess.check_output(
        [sys.executable, "-c", code], text=True, stderr=subprocess.STDOUT
    )
    info = json.loads(out.strip().splitlines()[-1])
except Exception as exc:
    print("\nPyTorch/CUDA check skipped (setup installs PyTorch):", exc)
else:
    print(
        f"GPU: {info['name']} | VRAM: {info['vram_gb']:.1f} GB | "
        f"compute capability: {info['cc'][0]}.{info['cc'][1]}"
    )
    print(f"PyTorch (before install): {info['torch']}")
    major, minor = info["cc"]
    if major < 7 or (major == 7 and minor < 5):
        raise SystemExit(
            "This GPU is too old for INT8 CUDA matmul. Use a T4, L4, A100, or newer."
        )
    if major < 8:
        print("\nNOTE: T4/Turing detected. The setup cell applies the float16 compatibility patch.")
    if info["vram_gb"] < 7:
        print("\nWARNING: less than 7 GB VRAM. Use the lowest-RAM settings and a smaller resolution.")
    elif info["vram_gb"] < 20:
        print("\nT4/16 GB GPU detected: keep the default INT8 + low-VRAM settings.")


In [ ]:
#@title 2. Upload your photo { display-mode: "form" }
import gc
import re
from pathlib import Path

from google.colab import files
from IPython.display import display
from PIL import Image, ImageOps

uploaded = files.upload()
if not uploaded:
    raise SystemExit("No file was selected. Re-run this cell and choose a photo.")
if len(uploaded) > 1:
    print("You selected more than one file; this notebook processes one photo per run. Using the first.")

raw_name = next(iter(uploaded))
raw_path = Path(raw_name)
if not raw_path.is_file():
    raw_path = Path("/content") / raw_name
    raw_path.write_bytes(uploaded[raw_name])

stem = Path(raw_name).stem
safe = re.sub(r"[^A-Za-z0-9._-]+", "_", stem).strip("._") or "photo"
PHOTO_NAME = safe
work = Path("/content/work") / PHOTO_NAME
work.mkdir(parents=True, exist_ok=True)

img = Image.open(raw_path)
img = ImageOps.exif_transpose(img)
try:
    if img.info.get("icc_profile"):
        import io
        from PIL import ImageCms
        srgb = ImageCms.createProfile("sRGB")
        src_profile = ImageCms.ImageCmsProfile(io.BytesIO(img.info["icc_profile"]))
        img = ImageCms.profileToProfile(img, src_profile, srgb, outputMode="RGB")
except Exception as exc:
    print("ICC profile conversion skipped:", exc)
img = img.convert("RGB")

PHOTO_SOURCE = work / "original_srgb.png"
img.save(PHOTO_SOURCE)
print("Saved:", PHOTO_SOURCE)
print("Resolution:", img.width, "x", img.height)
thumb = img.copy()
thumb.thumbnail((1024, 1024), Image.LANCZOS)
display(thumb)

del uploaded
gc.collect()


In [ ]:
#@title 3. Install CorridorKey 2.0 (first run ~10-15 min) { display-mode: "form" }
#@markdown Clones the pinned official repo, installs CUDA PyTorch, pins the C-RADIO/Transformers stack, adds the CUDA INT8 ConvRot path, and downloads the INT8 checkpoint plus foundation models.
#@markdown Default model is INT8 (about 1.0 GB main checkpoint). Selecting FP32 later downloads the 2.66 GB official checkpoint on demand.
#@markdown On T4/Turing GPUs it also applies the float16 compatibility patch (experimental).
force_reinstall = False #@param {type:"boolean"}

import hashlib
import json
import shutil
import subprocess
import sys
import urllib.request
from pathlib import Path

REPO_DIR = Path("/content/CorridorKey2.0")
REPO_URL = "https://github.com/nikopueringer/CorridorKey2.0.git"
REPO_COMMIT = "485f5e9ab759ead6138cc990776184b59d6ea971"
MARKER = Path("/content/.corridorkey_setup_v2.json")
TORCH_VERSION = "2.9.1"
TV_VERSION = "0.24.1"
PATCH_FILES = ("corridorkey/runtime.py", "corridorkey/video_io.py")
RESTORE_FILES = PATCH_FILES + ("corridorkey/blocks.py",)
PATCH_MARKER = REPO_DIR / ".t4_fp16_patch"
INT8_MARKER = REPO_DIR / ".int8_convrot_patch"

INT8_REPO = "boricuapab/corridorkey_v2_int8_convrot"
INT8_FILE = "corridorkey_v2_int8_convrot.safetensors"
INT8_REVISION = "8ee2f2130b9d254c01c735c354f3bc7cb71d56a8"
INT8_SHA256 = "9382f1b625a7d504340884682838437cdeeed3b3522036f02e1a47999a6bb20c"
INT8_LOADER_URL = (
    "https://raw.githubusercontent.com/boricuapab/ck2-mask-webui/"
    "001f71cc66b3f13adeac6b8e10f54b8d420bad8d/engine/corridorkey/int8_convrot.py"
)


def run(cmd, cwd=None):
    print("$", " ".join(str(c) for c in cmd), flush=True)
    subprocess.run([str(c) for c in cmd], cwd=str(cwd) if cwd else None, check=True)


def torch_state():
    code = (
        "import torch, torchvision; "
        "print(torch.__version__, torchvision.__version__, int(torch.cuda.is_available()))"
    )
    try:
        out = subprocess.check_output([sys.executable, "-c", code], text=True).strip().split()
        if len(out) >= 3:
            return out[0], out[1], out[2] == "1"
    except Exception:
        pass
    return None, None, False


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def replace_required(text, old, new, label):
    if new in text:
        return text
    if old not in text:
        raise RuntimeError(
            f"Could not apply {label}; upstream code changed. Expected fragment not found.")
    return text.replace(old, new, 1)


def patch_int8_runtime():
    loader = REPO_DIR / "corridorkey/int8_convrot.py"
    with urllib.request.urlopen(INT8_LOADER_URL, timeout=60) as response:
        loader.write_bytes(response.read())
    print("INT8 loader:", loader, flush=True)

    runtime = REPO_DIR / "corridorkey/runtime.py"
    text = runtime.read_text(encoding="utf-8")
    text = replace_required(
        text,
        '            "state_dict": load_file(str(path), device="cpu"),',
        '            "quantization": json.loads(header.get("corridorkey_quantization", "{}")),\n'
        '            "state_dict": load_file(str(path), device="cpu"),',
        "INT8 checkpoint metadata",
    )
    text = replace_required(
        text,
        '    model = CorridorKeyV3(**cfg)\n'
        '    missing, unexpected = model.load_state_dict(ckpt["state_dict"], strict=False)',
        '    model = CorridorKeyV3(**cfg)\n'
        '    quantization = ckpt.get("quantization", {})\n'
        '    if quantization:\n'
        '        from .int8_convrot import install_int8_linears\n'
        '        count = install_int8_linears(model, ckpt["state_dict"], quantization)\n'
        '        print(f"  INT8 ConvRot: {count} quantized linears; dynamic INT8 activations and INT8 CUDA matmul")\n'
        '    elif any(key.endswith(".comfy_quant") for key in ckpt["state_dict"]):\n'
        '        raise RuntimeError("Quantized weights are missing corridorkey_quantization metadata.")\n'
        '    missing, unexpected = model.load_state_dict(ckpt["state_dict"], strict=False)',
        "INT8 model installation",
    )
    runtime.write_text(text, encoding="utf-8")
    print("Patched runtime for INT8.", flush=True)

    blocks = REPO_DIR / "corridorkey/blocks.py"
    btext = blocks.read_text(encoding="utf-8")
    for old, new, label in (
        (
            "            ),\n        )\n\n    partial = torch.nonzero(~full",
            "            ).to(out.dtype),\n        )\n\n    partial = torch.nonzero(~full",
            "SDPA full-window dtype",
        ),
        (
            "            ),\n        )\n    if not groups:",
            "            ).to(out.dtype),\n        )\n    if not groups:",
            "SDPA grouped-window dtype",
        ),
    ):
        if new not in btext:
            if old not in btext:
                raise RuntimeError(f"Could not apply {label}; upstream code changed.")
            btext = btext.replace(old, new, 1)
    old = "out_nlhd[win_idx[:, None], token_idx[None, :]] = attn.permute(0, 2, 1, 3)"
    new = old + ".to(out.dtype)"
    if new not in btext:
        if old not in btext:
            raise RuntimeError("Could not apply indexed-write dtype fix; upstream code changed.")
        btext = btext.replace(old, new)
    blocks.write_text(btext, encoding="utf-8")
    print("Patched blocks for autocast dtype safety.", flush=True)
    INT8_MARKER.write_text("int8_convrot_v1\n", encoding="utf-8")


# Restore patched files before updating so git can move cleanly.
if (REPO_DIR / "infer.py").is_file() and (PATCH_MARKER.is_file() or INT8_MARKER.is_file()):
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "checkout", "--", *RESTORE_FILES],
        capture_output=True, text=True,
    )
    for marker in (PATCH_MARKER, INT8_MARKER):
        try:
            marker.unlink()
        except FileNotFoundError:
            pass
    print("Restored upstream files before updating.", flush=True)

# 1) Pinned repository
if not (REPO_DIR / "infer.py").is_file():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    run(["git", "clone", REPO_URL, str(REPO_DIR)])
def has_commit():
    return subprocess.run(
        ["git", "-C", str(REPO_DIR), "cat-file", "-e", REPO_COMMIT + "^{commit}"],
        capture_output=True,
    ).returncode == 0

if not has_commit():
    try:
        run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", REPO_COMMIT])
    except subprocess.CalledProcessError:
        print(
            "Could not fetch the pinned commit. Delete /content/CorridorKey2.0 and re-run setup.",
            flush=True,
        )
        raise
run(["git", "-C", str(REPO_DIR), "checkout", "-f", REPO_COMMIT])
commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
print("CorridorKey commit:", commit, flush=True)

marker = {}
if MARKER.is_file():
    try:
        marker = json.loads(MARKER.read_text())
    except Exception:
        marker = {}
same_setup = (not force_reinstall and marker.get("commit") == commit)

# 2) PyTorch
torch_v, tv_v, cuda_ok = torch_state()
need_torch = (
    force_reinstall
    or torch_v is None
    or not torch_v.startswith(TORCH_VERSION)
    or tv_v is None
    or not tv_v.startswith(TV_VERSION)
    or not cuda_ok
)
if need_torch:
    print(f"Installing PyTorch {TORCH_VERSION} + torchvision {TV_VERSION} (CUDA 12.8)...", flush=True)
    run([
        sys.executable, "-m", "pip", "install", "-q",
        f"torch=={TORCH_VERSION}", f"torchvision=={TV_VERSION}",
        "--index-url", "https://download.pytorch.org/whl/cu128",
    ])
else:
    print(f"PyTorch {torch_v} + torchvision {tv_v} already installed; skipping.", flush=True)

# 2b) INT8 loader + runtime patch (always re-applied)
patch_int8_runtime()

# 2c) T4/Turing compatibility: bfloat16 is not native on pre-Ampere GPUs.
try:
    cap_major = int(subprocess.check_output(
        [sys.executable, "-c", "import torch; print(torch.cuda.get_device_capability(0)[0])"],
        text=True,
    ).strip())
except Exception:
    cap_major = 8

if cap_major < 8:
    patched = []
    for rel in PATCH_FILES:
        p = REPO_DIR / rel
        text = p.read_text(encoding="utf-8")
        if "torch.bfloat16" in text:
            p.write_text(text.replace("torch.bfloat16", "torch.float16"), encoding="utf-8")
            patched.append(rel)
    PATCH_MARKER.write_text("fp16\n", encoding="utf-8")
    print("T4/Turing detected: applied the local float16 compatibility patch.", flush=True)
    print("Patched:", ", ".join(patched) if patched else "already patched", flush=True)
else:
    print("Ampere+ GPU detected: using the native bfloat16 path.", flush=True)

# 3) Python dependencies. Pin the C-RADIO / Transformers stack: Colab's
# preinstalled packages and Python 3.13 are not compatible with the old
# requirements.txt numpy<2 pin.
DEPS_CHECK_CODE = r'''
import importlib.metadata as md
import json
from packaging.version import Version

pinned = {
    "transformers": "4.57.6",
    "tokenizers": "0.22.2",
    "huggingface-hub": "0.36.2",
    "safetensors": "0.8.0",
    "timm": "1.0.30",
    "einops": "0.8.2",
}
minimum = {
    "numpy": "2.1",
    "opencv-python": "4.12",
    "scipy": "1.16",
    "pillow": "11",
    "tqdm": "4.67",
}
versions = {}
for name in list(pinned) + list(minimum):
    try:
        versions[name] = md.version(name)
    except md.PackageNotFoundError:
        versions[name] = None

ok = True
for name, want in pinned.items():
    got = versions.get(name)
    if got is None or Version(got) != Version(want):
        ok = False
for name, want in minimum.items():
    got = versions.get(name)
    if got is None or Version(got) < Version(want):
        ok = False

import_ok = True
import_error = None
try:
    from transformers import PreTrainedModel, PretrainedConfig
    from timm.models import VisionTransformer
    import cv2, safetensors, einops, scipy, utils3d, huggingface_hub
except Exception as exc:
    import_ok = False
    import_error = repr(exc)

print(json.dumps({
    "ok": bool(ok and import_ok),
    "versions": versions,
    "import_ok": import_ok,
    "import_error": import_error,
}))
'''


def check_runtime_deps():
    try:
        out = subprocess.check_output(
            [sys.executable, "-c", DEPS_CHECK_CODE], text=True, stderr=subprocess.STDOUT
        )
        data = json.loads(out.strip().splitlines()[-1])
    except Exception as exc:
        return False, {"error": repr(exc)}
    return bool(data.get("ok")), data


deps_ok, dep_info = check_runtime_deps()
if force_reinstall or not same_setup or not deps_ok:
    print("Installing pinned runtime dependencies...", flush=True)
    # Normal install for Transformers' own dependency chain; it does not pull torch.
    run([
        sys.executable, "-m", "pip", "install", "-q",
        "transformers==4.57.6",
        "tokenizers==0.22.2",
        "huggingface-hub==0.36.2",
        "safetensors==0.8.0",
    ])
    # Force the exact versions without touching CUDA torch/torchvision.
    run([
        sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps",
        "transformers==4.57.6",
        "tokenizers==0.22.2",
        "huggingface-hub==0.36.2",
        "safetensors==0.8.0",
        "timm==1.0.30",
        "einops==0.8.2",
    ])
    run([
        sys.executable, "-m", "pip", "install", "-q",
        "numpy>=2.1,<3",
        "opencv-python>=4.12",
        "scipy>=1.16",
        "pillow>=11",
        "tqdm>=4.67",
        "utils3d @ git+https://github.com/EasternJournalist/utils3d.git@83bfa5bda008803f6521ce9547f94023b8afeddd",
    ])
    deps_ok, dep_info = check_runtime_deps()
    if not deps_ok:
        print("Dependency check output:", dep_info, flush=True)
        raise RuntimeError(
            "Pinned runtime dependencies are still not usable. "
            "Check the transformers/tokenizers/timm versions and the import error above."
        )
else:
    print("Pinned runtime dependencies already OK; skipping.", flush=True)

versions = dep_info.get("versions", {})
print(
    "Runtime versions: "
    + " | ".join(f"{k}={v}" for k, v in versions.items() if v is not None),
    flush=True,
)

# 4) ffmpeg + ProRes encoder
if shutil.which("ffmpeg") is None:
    run(["apt-get", "-qq", "install", "-y", "ffmpeg"])
encoders = subprocess.check_output(
    ["ffmpeg", "-hide_banner", "-encoders"], text=True, errors="ignore"
)
if "prores_ks" not in encoders:
    raise RuntimeError(
        "This ffmpeg build has no prores_ks encoder. Re-run: apt-get install -y ffmpeg"
    )
print("ffmpeg + prores_ks OK", flush=True)

# 5) Foundation models + stats (skip the 2.66 GB FP32 main checkpoint)
from huggingface_hub import hf_hub_download

foundation_stats = REPO_DIR / "weights/foundation_stats.safetensors"
moge = REPO_DIR / "vfms/moge/model.pt"
cradio_dir = REPO_DIR / "vfms/cradio/C-RADIOv4-SO400M"
rvm_dir = REPO_DIR / "vfms/rvm"
missing_foundations = (
    force_reinstall
    or not foundation_stats.is_file()
    or not moge.is_file()
    or not cradio_dir.is_dir()
    or not any(cradio_dir.iterdir())
    or not rvm_dir.is_dir()
    or not any(rvm_dir.glob("*.npz"))
)
if missing_foundations:
    print("Downloading foundation models (~2-3 GB, resumable)...", flush=True)
    run(
        [sys.executable, str(REPO_DIR / "scripts/download_weights.py"), "--skip_corridorkey"],
        cwd=REPO_DIR,
    )
    if not foundation_stats.is_file():
        hf_hub_download(
            "nikopueringer/CorridorKey2.0",
            "foundation_stats.safetensors",
            local_dir=str(REPO_DIR / "weights"),
        )
else:
    print("Foundation models already present; skipping.", flush=True)

# 6) INT8 ConvRot checkpoint
int8_path = REPO_DIR / "weights" / INT8_FILE
need_int8 = force_reinstall or not int8_path.is_file()
if not need_int8:
    try:
        need_int8 = sha256_file(int8_path) != INT8_SHA256
    except Exception:
        need_int8 = True
if need_int8:
    if int8_path.is_file():
        int8_path.unlink()
    print("Downloading INT8 ConvRot checkpoint (~1.0 GB)...", flush=True)
    int8_path = Path(hf_hub_download(
        INT8_REPO, INT8_FILE, revision=INT8_REVISION,
        local_dir=str(REPO_DIR / "weights"),
    ))
digest = sha256_file(int8_path)
if digest != INT8_SHA256:
    raise RuntimeError(f"INT8 checkpoint checksum mismatch: {digest}")
print("INT8 checkpoint OK:", int8_path, flush=True)

# 7) Done
MARKER.write_text(json.dumps({
    "commit": commit,
    "torch": TORCH_VERSION,
    "torchvision": TV_VERSION,
    "int8_sha256": INT8_SHA256,
}))
print("\nSetup complete.", flush=True)
print("Repo:", REPO_DIR, flush=True)
print("Default weights: INT8 ConvRot (" + int8_path.name + ")", flush=True)
print(
    "If Colab shows a 'restart runtime' banner, you can ignore it: "
    "inference runs in a fresh subprocess.",
    flush=True,
)


In [ ]:
#@title Advanced settings (optional) { display-mode: "form" }
#@markdown Defaults are tuned for a free T4: INT8 weights, low-VRAM chunking, and a 2048 px long side.
model_precision = "INT8 ConvRot (lowest RAM)" #@param ["INT8 ConvRot (lowest RAM)", "FP32 (official)"]
#@markdown INT8 uses about 1 GB of weights. FP32 is 2.66 GB and may OOM on a T4.
max_long_side = "2048" #@param ["2048", "3072", "4096", "0"]
#@markdown Longest output side. 0 = native resolution (may OOM on a T4).
use_low_vram = True #@param {type:"boolean"}
#@markdown Enables staged inference and attention/MLP chunking. Keep on for large photos.
foundation_chunk = 2 #@param {type:"integer"}
#@markdown Frames per foundation precompute chunk. Lower = less VRAM, slower.
stage2_batch = 1 #@param {type:"integer"}
#@markdown Full-resolution frames per stage-2 batch. Keep at 1 for low memory.
mlp_chunk_tokens = 32768 #@param {type:"integer"}
#@markdown Token chunk for MLP layers in low-VRAM mode.
swin_window_batch = 8 #@param {type:"integer"}
#@markdown Window batch for Swin attention in low-VRAM mode.
attn_query_chunk_tokens = 1024 #@param {type:"integer"}
#@markdown Query chunk for full attention in low-VRAM mode.
temp_dir = "/content/ck_temp" #@param {type:"string"}
#@markdown Scratch folder for low-VRAM feature memmaps. Keep it on /content, not /tmp.

save_alpha = False #@param {type:"boolean"}
#@markdown Also save the black/white matte.
save_fg = False #@param {type:"boolean"}
#@markdown Also save the raw foreground RGB (no alpha).
save_checker = False #@param {type:"boolean"}
#@markdown Also save a full-resolution foreground-over-checker render.
hint_quality = 0.95 #@param {type:"slider", min:0.0, max:1.0, step:0.05}
#@markdown Trust in the optional matte: 0.95 = clean artist matte, 0.75 = rough, <=0.5 = loose. Ignored without a matte.
despill_strength = 0.5 #@param {type:"slider", min:0.0, max:1.0, step:0.05}
#@markdown Green/blue spill removal in the cutout. 0 = none, 1 = full.
cutout_linear = False #@param {type:"boolean"}
#@markdown Export straight scene-linear RGB instead of sRGB. For linear VFX compositing only; normal viewers will look wrong.


In [ ]:
#@title 4. Optional: upload a first-frame matte { display-mode: "form" }
#@markdown A matte is a black/white image: **white = subject**, **black = background**. If the image has an alpha channel, the alpha is used.
#@markdown For a single photo this is effectively a full hint. If you don't have one, leave this off.
ADD_HINT = False #@param {type:"boolean"}

from pathlib import Path

from google.colab import files
from IPython.display import display
from PIL import Image

if "PHOTO_NAME" not in globals():
    raise SystemExit("Run the photo upload cell first.")

HINT_SOURCE = None
if ADD_HINT:
    uploaded_hint = files.upload()
    if not uploaded_hint:
        print("No hint file selected; continuing without a hint.")
    else:
        hint_name = next(iter(uploaded_hint))
        hint_path = Path(hint_name)
        if not hint_path.is_file():
            hint_path = Path("/content") / hint_name
            hint_path.write_bytes(uploaded_hint[hint_name])
        HINT_SOURCE = Path("/content/work") / PHOTO_NAME / "hint_source.png"
        try:
            hint_img = Image.open(hint_path)
        except Exception as exc:
            raise SystemExit(f"Could not read the hint image: {exc}")
        hint_img.save(HINT_SOURCE)
        print(f"Hint saved: {HINT_SOURCE} ({hint_img.width} x {hint_img.height}, mode {hint_img.mode})")
        thumb = hint_img.copy()
        thumb.thumbnail((1024, 1024), Image.LANCZOS)
        display(thumb)
else:
    print("No hint will be used. The model will choose the foreground subject.")


In [ ]:
#@title 5. Remove the background { display-mode: "form" }
import gc
import os
import resource
import shutil
import subprocess
import sys
import time
from pathlib import Path

from PIL import Image

if "PHOTO_SOURCE" not in globals() or not Path(PHOTO_SOURCE).is_file():
    raise SystemExit("Run the photo upload cell first.")

save_alpha = bool(globals().get("save_alpha", False))
save_fg = bool(globals().get("save_fg", False))
save_checker = bool(globals().get("save_checker", False))
hint_quality = float(globals().get("hint_quality", 0.95))
despill_strength = float(globals().get("despill_strength", 0.5))
max_long_side = int(globals().get("max_long_side", 2048))
cutout_linear = bool(globals().get("cutout_linear", False))
model_precision = str(globals().get("model_precision", "INT8 ConvRot (lowest RAM)"))
use_low_vram = bool(globals().get("use_low_vram", True))
foundation_chunk = max(1, int(globals().get("foundation_chunk", 2)))
stage2_batch = max(1, int(globals().get("stage2_batch", 1)))
mlp_chunk_tokens = max(0, int(globals().get("mlp_chunk_tokens", 32768)))
swin_window_batch = max(0, int(globals().get("swin_window_batch", 8)))
attn_query_chunk_tokens = max(0, int(globals().get("attn_query_chunk_tokens", 1024)))
temp_dir = str(globals().get("temp_dir", "/content/ck_temp"))

REPO_DIR = Path(globals().get("REPO_DIR", "/content/CorridorKey2.0"))
infer_py = REPO_DIR / "infer.py"
if not infer_py.is_file():
    raise SystemExit("CorridorKey is not installed. Run the setup cell above.")

use_int8 = model_precision.startswith("INT8")
weights = REPO_DIR / "weights" / (
    "corridorkey_v2_int8_convrot.safetensors" if use_int8 else "corridorkey_v2.safetensors"
)
if not weights.is_file():
    if use_int8:
        raise SystemExit("INT8 weights are missing. Re-run the setup cell.")
    print("FP32 checkpoint is missing; downloading ~2.66 GB...")
    from huggingface_hub import hf_hub_download
    weights = Path(hf_hub_download(
        "nikopueringer/CorridorKey2.0",
        "corridorkey_v2.safetensors",
        local_dir=str(REPO_DIR / "weights"),
    ))

work = Path("/content/work") / PHOTO_NAME
input_dir = work / "input"
input_dir.mkdir(parents=True, exist_ok=True)
input_path = input_dir / "0001.png"

img = Image.open(PHOTO_SOURCE).convert("RGB")
original_size = img.size
if max_long_side and max(img.size) > max_long_side:
    scale = max_long_side / max(img.size)
    new_size = (max(1, round(img.width * scale)), max(1, round(img.height * scale)))
    img = img.resize(new_size, Image.LANCZOS)
    print(f"Resized input from {original_size} to {new_size}.")
else:
    print(f"Input resolution: {img.width} x {img.height}")
img.save(input_path)

hint_path = None
hint_src = globals().get("HINT_SOURCE")
if hint_src and Path(hint_src).is_file():
    hint = Image.open(hint_src)
    hint = hint.convert("RGBA") if hint.mode in ("RGBA", "LA") else hint.convert("L")
    if hint.size != img.size:
        hint = hint.resize(img.size, Image.LANCZOS)
        print(f"Resized hint to the input size ({img.width} x {img.height}).")
    hint_path = work / "hint.png"
    hint.save(hint_path)
    print("Using hint:", hint_path)
del img

outputs = ["cutout"]
if save_alpha:
    outputs.append("alpha")
if save_fg:
    outputs.append("fg")
if save_checker:
    outputs.append("checker")

out_dir = Path("/content/outputs") / PHOTO_NAME
if out_dir.exists():
    shutil.rmtree(out_dir)
out_dir.mkdir(parents=True, exist_ok=True)
Path(temp_dir).mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, str(infer_py),
    "--input", str(input_dir),
    "--output_dir", str(out_dir),
    "--outputs", *outputs,
    "--frame_dir_fps", "24",
    "--hint_quality", str(hint_quality),
    "--despill_strength", str(despill_strength),
    "--weights", str(weights),
    "--temp_dir", temp_dir,
    "--stage2_batch", str(stage2_batch),
    "--low_vram_foundation_chunk", str(foundation_chunk),
    "--low_vram_mlp_chunk_tokens", str(mlp_chunk_tokens),
    "--low_vram_swin_window_batch", str(swin_window_batch),
    "--low_vram_full_attn_query_chunk_tokens", str(attn_query_chunk_tokens),
]
if use_low_vram:
    cmd.append("--low_vram")
if hint_path is not None:
    cmd += ["--hint_first_frame", str(hint_path)]
if cutout_linear:
    cmd.append("--cutout_linear")

print("\nModel:", model_precision)
print("Command:")
print(" ".join(cmd))
print("Output directory:", out_dir)
print()
gc.collect()
start = time.time()
env = os.environ.copy()
env.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
env["USE_TF"] = "0"
env["USE_FLAX"] = "0"
env["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
proc = subprocess.run(cmd, text=True, capture_output=True, env=env)
if proc.stdout:
    print(proc.stdout)
if proc.stderr:
    print(proc.stderr, file=sys.stderr)
if proc.returncode != 0:
    print("\nInference failed with exit code", proc.returncode)
    print("Common fixes:")
    print("- CUDA out of memory: keep INT8, set max_long_side to 2048, keep use_low_vram on.")
    print("- Still out of memory: set max_long_side to 2048 or lower, or use a GPU with more VRAM.")
    print("- System RAM out of memory: restart the runtime and re-run setup before inference.")
    print("- Missing weights: re-run the setup cell.")
    raise SystemExit(proc.returncode)
peak_kb = resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss
print(f"\nDone in {time.time() - start:.1f} s.")
print(f"Peak child system RAM: {peak_kb / 1024 / 1024:.2f} GB")
print("Outputs are in:", out_dir)

OUT_DIR = out_dir
CLIP_NAME = PHOTO_NAME


In [ ]:
#@title 6. Preview & download { display-mode: "form" }
import subprocess
import zipfile
from pathlib import Path

import numpy as np
from IPython.display import FileLink, Image as IPImage, display
from PIL import Image

if "OUT_DIR" not in globals():
    raise SystemExit("Run the inference cell first.")

out_dir = Path(OUT_DIR)
clip = CLIP_NAME


def human(n):
    n = float(n)
    for unit in ("B", "KB", "MB", "GB"):
        if n < 1024 or unit == "GB":
            return f"{n:.1f} {unit}"
        n /= 1024


def checker_rgba(width, height, tile=32):
    yy, xx = np.indices((height, width))
    mask = (((xx // tile) + (yy // tile)) % 2).astype(np.uint8)
    gray = np.where(mask == 1, 210, 160).astype(np.uint8)
    rgb = np.repeat(gray[:, :, None], 3, axis=2)
    return Image.fromarray(rgb, "RGB").convert("RGBA")


def show_image(path, max_side=1024):
    img = Image.open(path)
    img.thumbnail((max_side, max_side), Image.LANCZOS)
    display(img)


def extract_first_frame(video_path, png_path):
    subprocess.run(
        ["ffmpeg", "-y", "-loglevel", "error", "-i", str(video_path),
         "-frames:v", "1", "-pix_fmt", "rgba", str(png_path)],
        check=True,
    )


items = []
cutout_mov = out_dir / f"{clip}_cutout.mov"
cutout_png = out_dir / f"{clip}_cutout.png"

if cutout_mov.is_file():
    extract_first_frame(cutout_mov, cutout_png)
    rgba = Image.open(cutout_png).convert("RGBA")
    bg = checker_rgba(rgba.width, rgba.height)
    preview = Image.alpha_composite(bg, rgba).convert("RGB")
    preview_path = out_dir / f"{clip}_checker_preview.png"
    preview.save(preview_path)
    print("Transparent PNG:", cutout_png, f"({human(cutout_png.stat().st_size)})")
    print("Resolution:", rgba.width, "x", rgba.height)
    print("Alpha range:", rgba.getchannel("A").getextrema())
    if globals().get("cutout_linear", False):
        print("NOTE: cutout_linear was enabled; PNG values are scene-linear, not sRGB.")
    print("\nResult over a checkerboard:")
    show_image(preview_path, 1024)
    items.extend([cutout_png, preview_path])
else:
    print("No cutout output found. Check the inference log above.")

for name, filename in (("alpha", f"{clip}_alpha.mp4"), ("fg", f"{clip}_fg.mp4")):
    video = out_dir / filename
    if video.is_file():
        png = out_dir / f"{clip}_{name}.png"
        extract_first_frame(video, png)
        print(f"{name}: {png} ({human(png.stat().st_size)})")
        show_image(png, 512)
        items.append(png)

for video in sorted(out_dir.glob(f"{clip}_checker_comp_despill*.mp4")):
    png = out_dir / f"{clip}_checker.png"
    extract_first_frame(video, png)
    print("checker:", png, f"({human(png.stat().st_size)})")
    show_image(png, 512)
    items.append(png)

original = Path(PHOTO_SOURCE)
if original.is_file():
    items.append(original)

unique = []
seen = set()
for p in items:
    p = Path(p)
    if p.is_file() and str(p) not in seen:
        unique.append(p)
        seen.add(str(p))

zip_path = out_dir / f"{clip}_result.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in unique:
        zf.write(p, p.name)
print("\nZIP:", zip_path, f"({human(zip_path.stat().st_size)})")

if cutout_png.is_file():
    display(FileLink(str(cutout_png), result_html_prefix="Download transparent PNG: "))
display(FileLink(str(zip_path), result_html_prefix="Download ZIP: "))


<details>
<summary><b>⚙️ Parameter reference — what each advanced setting does</b></summary>

The transparent `cutout` is always written. Everything below is optional.

| Setting | Default | Effect |
|---|---|---|
| `model_precision` | INT8 | **INT8 ConvRot** uses about 1.0 GB of weights and is the low-RAM CUDA path. **FP32** downloads the 2.66 GB official checkpoint; it needs more memory. |
| `max_long_side` | 2048 | Resize the input before inference so its longest side is at most this many pixels. `0` keeps the original and may OOM on a T4. The output matches the resized input. |
| `use_low_vram` | on | Enables staged inference and attention/MLP chunking. Keep on for large photos. |
| `foundation_chunk` | 2 | Frames per foundation precompute chunk. Lower = less VRAM, slower. |
| `stage2_batch` | 1 | Full-resolution frames per stage-2 batch. Keep at 1 for low memory. |
| `mlp_chunk_tokens` | 32768 | Token chunk for MLP layers in low-VRAM mode. |
| `swin_window_batch` | 8 | Window batch for Swin attention in low-VRAM mode. |
| `attn_query_chunk_tokens` | 1024 | Query chunk for full attention in low-VRAM mode. |
| `temp_dir` | `/content/ck_temp` | Scratch folder for low-VRAM feature memmaps. Keep it on `/content`, not `/tmp`. |
| `save_alpha` | off | Also save the black/white matte as `..._alpha.png`. |
| `save_fg` | off | Also save the raw foreground RGB as `..._fg.png` (no alpha). |
| `save_checker` | off | Also save the model's full-resolution foreground-over-checker render. |
| `hint_quality` | 0.95 | How much to trust an uploaded matte. 0.95 = clean artist matte; 0.75 = rough automated mask; ≤0.5 = loose/scribble. Ignored when no matte is uploaded. |
| `despill_strength` | 0.5 | Removes green/blue spill from the cutout. 0 = none, 1 = full. |
| `cutout_linear` | off | Writes straight scene-linear RGB instead of sRGB. For linear VFX compositing only; normal image viewers will look wrong. |

**Hints.** A matte is a black/white image: white means subject, black means background. If the image has an alpha channel, the alpha is used. The notebook resizes the matte to the photo automatically. For one photo, a first-frame hint is effectively a full hint.

**INT8 note.** The INT8 checkpoint is a community W8A8 ConvRot conversion. It changes numerical results, especially localized alpha, compared with FP32. If quality matters more than memory, select `FP32 (official)` and a smaller `max_long_side`.

**What the notebook runs.** With defaults:

```bash
python infer.py \
  --input /content/work/<name>/input \
  --output_dir /content/outputs/<name> \
  --outputs cutout [alpha] [fg] [checker] \
  --frame_dir_fps 24 \
  --hint_quality 0.95 \
  --despill_strength 0.5 \
  --weights weights/corridorkey_v2_int8_convrot.safetensors \
  --low_vram --stage2_batch 1 \
  --low_vram_foundation_chunk 2 \
  --temp_dir /content/ck_temp \
  [--hint_first_frame /content/work/<name>/hint.png] \
  [--cutout_linear]
```

The input is resized to `max_long_side` before inference. The model pads internally and crops back to that size.

</details>


<details>
<summary><b>🛠️ Troubleshooting</b></summary>

**No GPU / “requires a CUDA GPU”.** In Colab: *Runtime → Change runtime type → Hardware accelerator → GPU*, then re-run the check cell.

**Colab asks to restart after installing PyTorch.** You can ignore it: inference runs in a fresh subprocess with the new PyTorch. If you do restart, re-run from the GPU check; the setup cell skips finished work.

**CUDA out of memory.** Keep `model_precision = INT8`, keep `use_low_vram = on`, and set `max_long_side = 2048`. If it still fails, try 1536 or 1024. A high-RAM Colab runtime also helps.

**System RAM out of memory / session crash.** This is different from CUDA OOM. Restart the runtime, re-run setup, and keep `max_long_side` at 2048. The setup cell downloads the 1.0 GB INT8 checkpoint instead of the 2.66 GB FP32 one by default.

**T4 / bfloat16.** Pre-Ampere GPUs (T4/Turing) lack native bfloat16. The setup cell automatically patches the runtime to float16 and prints a notice. This is experimental; if you hit a CUDA error, switch to an A100/L4 runtime.

**The run seems slow.** The first run loads the model and downloads weights. A T4 is slow; A100/L4 is much faster. The photo path only processes one frame, so the model load dominates.

**The session disconnected.** `/content` is erased when the session ends. Download the PNG/ZIP before disconnecting. The notebooks intentionally do not use Google Drive.

**The subject is wrong.** Upload a first-frame matte in the optional cell: white = subject, black = background, then re-run inference. Or use the CLI locally with a full hint.

**Edges have green/blue spill.** Raise `despill_strength` toward 1.0. For a stronger result, provide a cleaner hint.

**INT8 edges look different from FP32.** W8A8 quantization can change localized alpha. Switch to `FP32 (official)` and lower `max_long_side` if quality matters more than memory.

**My photo is rotated.** The notebook applies EXIF orientation automatically. If it still looks wrong, rotate the source image before uploading.

**I need HDR / scene-linear output.** This notebook extracts an 8-bit sRGB PNG from the model's ProRes 4444 cutout. For unclipped linear EXR, use `infer.py --output_format exr` from the CLI; see the repo README.

**My image has a Log / ACES / HDR profile.** The runtime expects sRGB/Rec.709-like display-coded input. Convert upstream to sRGB before uploading.

**The setup patch fails after an upstream change.** The notebook pins the official repo to a known commit. If you edit `REPO_COMMIT`, the INT8 runtime patch may need updating.

</details>


<details>
<summary><b>📜 License & acknowledgements</b></summary>

CorridorKey 2.0 was designed and trained by **Zachary Harris**, in direct collaboration with **Niko** and **Corridor Digital**.

- You may use CorridorKey and its outputs for personal or commercial work. What you make is yours.
- You may not sell the tool, its code, or its model, or offer it as a paid API/inference service, without permission from Corridor Digital.
- If you build on the code or model, your release must be freely accessible, stay under the same license, and keep the CorridorKey name. Donationware/pay-what-you-want is fine.
- Organizations with more than US $20 million in annual revenue need to purchase a CorridorKey plugin seat for each person using the tool.
- Commercial software integrations: contact contact@corridordigital.com.

Full terms: [LICENSE](https://github.com/nikopueringer/CorridorKey2.0/blob/master/LICENSE). Official plugin/app: [corridorkey.com](https://corridorkey.com).

CorridorKey builds on three frozen foundation models:

- **C-RADIO** by NVIDIA — NVIDIA Open Model License
- **MoGe** by Microsoft — MIT
- **RVM** by Google DeepMind — code Apache 2.0, weights CC-BY 4.0

The optional INT8 ConvRot checkpoint is a community conversion by **boricuapab** (`boricuapab/corridorkey_v2_int8_convrot`). The loader is from `boricuapab/ck2-mask-webui`. The notebooks download these at runtime and keep the original CorridorKey license and attribution. The model weights are not redistributed by this repository.

Details are in [THIRD_PARTY_NOTICES.md](https://github.com/nikopueringer/CorridorKey2.0/blob/master/THIRD_PARTY_NOTICES.md).

</details>
